In [1]:
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

notebook_name = "Bronze"
run_start = datetime.utcnow()
logger.info(f"[START] {notebook_name} — {run_start.strftime('%Y-%m-%d %H:%M:%S')} UTC")

StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 3, Finished, Available, Finished, False)

INFO:__main__:[START] Bronze — 2026-04-24 03:03:00 UTC


In [2]:
# Run this at the top of each notebook to see what tables it touches
# Paste the output for all 4 notebooks

spark.sql("SHOW TABLES").show(50, truncate=False)

StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 4, Finished, Available, Finished, False)

+-----------------------------------------------------------------+-----------------------------+-----------+
|namespace                                                        |tableName                    |isTemporary|
+-----------------------------------------------------------------+-----------------------------+-----------+
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|bronze_vehicle_images        |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|complaints                   |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_brand_reliability       |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_component_failures      |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_dim_failure_category    |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_dim_make                |false      |
|Vehicle_r

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name, lit

spark

df_bronze_complaints = spark.read.parquet("Files/Bronze/raw/complaints_raw", header= True, inferSchema = True)
df_bronze_recalls = spark.read.parquet("Files/Bronze/raw/recalls_raw", header= True, inferSchema = True)
df_bronze_logos = spark.read.parquet("Files/Bronze/raw/logos_raw", header= True, inferSchema = True)
df_bronze_vehicle_images = spark.read.csv("Files/Bronze/raw/vehicle_images", header= True, inferSchema = True)
df_bronze_vehicle_repair = spark.read.csv("Files/Bronze/raw/vehicle_repair_dataset", header= True, inferSchema = True)



print(f"complaints row count: {df_bronze_complaints.count()}")
print(f"recalls row count: {df_bronze_recalls.count()}")
print(f"logos row count: {df_bronze_logos.count()}")
print(f"vehicle imahes row count: {df_bronze_vehicle_images.count()}")
print(f"vehicle repair row count: {df_bronze_vehicle_repair.count()}")


df_bronze_complaints.printSchema
df_bronze_recalls.printSchema
df_bronze_logos.printSchema
df_bronze_vehicle_images.printSchema
df_bronze_vehicle_repair.printSchema

StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 5, Finished, Available, Finished, False)

complaints row count: 178648
recalls row count: 8061
logos row count: 27
vehicle imahes row count: 1102
vehicle repair row count: 2662


<bound method DataFrame.printSchema of DataFrame[make: string, model: string, year: int, avg_annual_repair_cost: int, maintenance_frequency: double, repair_category: string]>

In [4]:
# Add the Metadata Columns

df_bronze_complaints_enriched = df_bronze_complaints \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("sourcefile", input_file_name()) \
    .withColumn("layer", lit("Bronze"))


df_bronze_recalls_enriched = df_bronze_recalls \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("sourcefile", input_file_name()) \
    .withColumn("layer", lit("Bronze"))


df_bronze_logos_enriched = df_bronze_logos \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("sourcefile", input_file_name()) \
    .withColumn("layer", lit("Bronze"))


df_bronze_vehicle_images_enriched = df_bronze_vehicle_images \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("sourcefile", input_file_name()) \
    .withColumn("layer", lit("Bronze"))


df_bronze_vehicle_repair_enriched = df_bronze_vehicle_repair \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("sourcefile", input_file_name()) \
    .withColumn("layer", lit("Bronze"))





StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 6, Finished, Available, Finished, False)

In [5]:
# Writing as Delta table 

df_bronze_complaints_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("complaints")


df_bronze_recalls_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("recalls")


df_bronze_logos_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("logos")



df_bronze_vehicle_images_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("vehicle_images")


df_bronze_vehicle_repair_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("maintenance_data")    




print("Delta tables created successfullly")            

StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 7, Finished, Available, Finished, False)

Delta tables created successfullly


In [6]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, upper, explode, split

# Load your raw complaints data — adjust table name if different
complaints_raw = spark.read.format("delta").load("Tables/dbo/complaints")  

# Define the clean category mapping
# Map NHTSA raw component strings to clean display categories
category_map = {
    "ENGINE": "Engine",
    "ENGINE AND ENGINE COOLING": "Engine",
    "POWER TRAIN": "Powertrain",
    "ELECTRICAL SYSTEM": "Electrical",
    "STEERING": "Steering",
    "SERVICE BRAKES": "Brakes",
    "AIR BAGS": "Air Bags",
    "FUEL/PROPULSION SYSTEM": "Fuel System",
    "FUEL SYSTEM, GASOLINE": "Fuel System",
    "FORWARD COLLISION AVOIDANCE": "Safety Systems",
    "VEHICLE SPEED CONTROL": "Safety Systems",
    "VISIBILITY/WIPER": "Visibility",
    "VISIBILITY": "Visibility",
    "EXTERIOR LIGHTING": "Lighting",
    "STRUCTURE": "Structure",
    "UNKNOWN OR OTHER": "Other",
}

# Build a broadcast-friendly map expression
mapping_expr = F.create_map(
    *[item for pair in 
      [(F.lit(k), F.lit(v)) for k, v in category_map.items()] 
      for item in pair]
)

# Load silver complaints — adjust column names to match yours
silver_complaints = spark.read.format("delta").load("Tables/dbo/silver_complaints")

# Check your column names first
silver_complaints.printSchema()

StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 8, Finished, Available, Finished, False)

root
 |-- complaint_id: string (nullable = true)
 |-- vehicle_make: string (nullable = true)
 |-- vehicle_model: string (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- issue_description: string (nullable = true)
 |-- dateOfIncident: string (nullable = true)
 |-- layer: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- dataOfIncident: date (nullable = true)
 |-- silver_timestamp: timestamp (nullable = true)



In [7]:
# Data Exploration

from pyspark.sql.functions import col, count, isnan, when, sum

print("=== NULL CHECK ===")

df_bronze_complaints_enriched.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze_complaints_enriched.columns
]).show()


df_bronze_recalls_enriched.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze_recalls_enriched.columns
]).show()


df_bronze_logos_enriched.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze_logos_enriched.columns
]).show()


df_bronze_vehicle_images_enriched.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze_vehicle_images_enriched.columns
]).show()


df_bronze_vehicle_repair_enriched.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze_vehicle_repair_enriched.columns
]).show()








StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 9, Finished, Available, Finished, False)

=== NULL CHECK ===
+------------+------------+-------------+----------+---------+-----------------+--------------+-------------------+----------+-----+
|complaint_id|vehicle_make|vehicle_model|model_year|component|issue_description|dateOfIncident|ingestion_timestamp|sourcefile|layer|
+------------+------------+-------------+----------+---------+-----------------+--------------+-------------------+----------+-----+
|           0|           0|            0|         0|        0|                0|             0|                  0|         0|    0|
+------------+------------+-------------+----------+---------+-----------------+--------------+-------------------+----------+-----+

+---------+------------+-------------+----------+------------+-----------+-----------+---------+--------------+-------------------+----------+-----+
|recall_id|vehicle_make|vehicle_model|model_year|recall_issue|recall_date|Consequence|Component|affected_units|ingestion_timestamp|sourcefile|layer|
+---------+------

In [8]:
print("=== DUPLICATE CHECK ===")

total_complaints = df_bronze_complaints_enriched.count()
distinct = df_bronze_complaints.dropDuplicates(["complaint_id"]).count()
print(f"Total rows : {total_complaints} | Distinct complaint_id: {distinct} | Duplicates: {total_complaints - distinct} ")


total_recalls = df_bronze_recalls_enriched.count()
distinct = df_bronze_recalls.dropDuplicates(["recall_id"]).count()
print(f"Total rows : {total_recalls} | Distinct recall_id: {distinct} | Duplicates: {total_recalls - distinct} ")



# deal with the recall after deleting



total_logos = df_bronze_logos_enriched.count()
distinct = df_bronze_logos.dropDuplicates(["vehicle_make"]).count()
print(f"Total rows : {total_logos} | Distinct vehicle_make: {distinct} | Duplicates: {total_logos - distinct} ")



total_vehicle_images = df_bronze_vehicle_images_enriched.count()
distinct = df_bronze_vehicle_images.dropDuplicates(["make"]).count()
print(f"Total rows : {total_vehicle_images} | Distinct make: {distinct} | Duplicates: {total_vehicle_images - distinct} ")


total_vehicle_repair = df_bronze_vehicle_repair_enriched.count()
distinct = df_bronze_vehicle_repair.dropDuplicates(["make"]).count()
print(f"Total rows : {total_vehicle_repair} | Distinct make: {distinct} | Duplicates: {total_vehicle_repair - distinct} ")


StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 10, Finished, Available, Finished, False)

=== DUPLICATE CHECK ===
Total rows : 178648 | Distinct complaint_id: 176374 | Duplicates: 2274 
Total rows : 8061 | Distinct recall_id: 1680 | Duplicates: 6381 
Total rows : 27 | Distinct vehicle_make: 27 | Duplicates: 0 
Total rows : 1102 | Distinct make: 14 | Duplicates: 1088 
Total rows : 2662 | Distinct make: 27 | Duplicates: 2635 


In [9]:
# ── Pipeline run completion log ──────────────────────────────────
run_end = datetime.utcnow()
duration = (run_end - run_start).seconds
logger.info(f"[END] {notebook_name} — completed in {duration}s")

# Write run log to lakehouse for monitoring
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

log_schema = StructType([
    StructField("notebook_name",  StringType(),   True),
    StructField("run_start",      TimestampType(), True),
    StructField("run_end",        TimestampType(), True),
    StructField("duration_secs",  IntegerType(),  True),
    StructField("status",         StringType(),   True),
])

log_row = [Row(
    notebook_name = notebook_name,
    run_start     = run_start,
    run_end       = run_end,
    duration_secs = duration,
    status        = "SUCCESS"
)]

log_df = spark.createDataFrame(log_row, schema=log_schema)

log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pipeline_run_log")

print(f"[DONE] {notebook_name} — {duration}s")

StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 11, Finished, Available, Finished, False)

INFO:__main__:[END] Bronze — completed in 66s
INFO:trident_token_library_wrapper:Calling getAccessToken from PyTridentTokenLibrary


[DONE] Bronze — 66s


In [11]:
spark.read.format("delta").load("Tables/dbo/pipeline_run_log").show(truncate=False)

StatementMeta(, 6ec08344-c3a8-460d-b5b0-e2e3f506a61e, 13, Finished, Available, Finished, False)

+----------------+--------------------------+--------------------------+-------------+-------+
|notebook_name   |run_start                 |run_end                   |duration_secs|status |
+----------------+--------------------------+--------------------------+-------------+-------+
|Bronze_ingestion|2026-04-24 02:32:21.73143 |2026-04-24 02:54:46.748086|1345         |SUCCESS|
|Bronze          |2026-04-24 03:03:00.139947|2026-04-24 03:04:06.873314|66           |SUCCESS|
|NULL            |2026-04-23 16:56:04.262779|2026-04-23 17:22:49.896034|1605         |SUCCESS|
|NULL            |2026-04-23 19:06:23.265507|2026-04-23 19:26:40.750176|1217         |SUCCESS|
|NULL            |2026-04-23 17:36:41.728831|2026-04-23 17:39:29.644403|167          |SUCCESS|
|NULL            |2026-04-23 16:49:15.708794|2026-04-23 16:50:55.963152|100          |SUCCESS|
|NULL            |2026-04-23 17:28:59.379131|2026-04-23 17:30:45.796476|106          |SUCCESS|
|NULL            |2026-04-23 16:51:42.900625|2026-